# Relatório dos modelos campeões (gráficos por UF, região e Brasil)

Junta, para cada **UF, macrorregião e Brasil**:
- **semanal (ENS_stack_sem):** o que o campeão previu 1 semana antes em todo o histórico de avaliação (2018–2026) e o leque das **próximas 4 semanas**;
- **temporada (ENS_pico7_simples_UF):** a previsão de cada temporada feita em junho (2018/19 → a mais recente).

**De onde vêm as previsões** (o que existir primeiro):
1. as saídas dos notebooks de produção (`producao_semanal/`, `producao_temporada/`, inclusive os backtests);
2. as pastas da pesquisa no Drive (`resultados_semanal*`, `resultados_uf_r7/`).

**Saída:** `relatorio_campeoes/modelos_campeoes.html` (autocontido; abre em qualquer navegador) e os PNG em `relatorio_campeoes/figuras/`. CPU, ~10 min.


In [ ]:
PACOTES = ''

# @title 1 · Área de trabalho (Drive no Colab ou pasta local), pacotes e contexto
import os, sys, shutil, glob, json, time, subprocess, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/prod', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False          # Jupyter aberto na área de trabalho
if NO_COLAB:
    if PACOTES:
        subprocess.run(f'pip install -q {PACOTES}', shell=True, check=True)
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py', 'r4_lib.py',
                 'r5_lib.py', 'r6_lib.py', 'r7_lib.py', 'prod_lib.py', 'relatorio_campeoes.py', 'graficos_previsoes.py',
                 'dados', 'dados_uf', 'referencias_imdc3', 'parametros'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou {item} em {PASTA_DRIVE}: rode scripts/04_montar_area_de_trabalho.py e suba a pasta'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
sys.path.insert(0, PASTA_LOCAL)
import importlib, v4_lib as V, comp_lib as L, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3, prod_lib as PL
for m in (V, L, U, S, R2, R3, PL):
    importlib.reload(m)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 120)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'), os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
P = {k: os.path.join(PASTA_DRIVE, v) for k, v in dict(PS='producao_semanal', PT='producao_temporada', REL='relatorio_campeoes',
                                                      CACHE='cache_producao').items()}
for p in P.values():
    os.makedirs(p, exist_ok=True)
PAR = PL.ler_parametros(os.path.join(PASTA_LOCAL, 'parametros'))
ULTIMA = pd.Timestamp(D.datas[D.ultimo_obs]).date()
print('última semana observada:', ULTIMA, '· UFs:', D.N)


In [ ]:
# @title 2 · Previsões do campeão semanal: histórico (h = 1…4) e próximas semanas
import graficos_previsoes as G, relatorio_campeoes as RC
importlib.reload(G); importlib.reload(RC)
MEMB = list(next(iter(PAR['pesos_stack_semanal'].values())).keys())
pastas = [os.path.join(P['PS'], 'backtest')] + sorted(p for p in glob.glob(os.path.join(PASTA_DRIVE, 'resultados_semanal*'))
                                                     if os.path.isdir(p) and not p.endswith('_avaliacao'))
P0 = S.ler_semanal([p for p in pastas if os.path.isdir(p)])
P0 = P0[P0.modelo.isin(MEMB) & P0.rodada.str.match(r'^s\d{4}$')].copy()
P0['date'], P0['origem'] = pd.to_datetime(P0['date']), pd.to_datetime(P0['origem'])
HIST = PL.ensembles_semanais(P0, D, PAR) if len(P0) else P0
HIST = HIST[HIST.modelo == PL.CAMPEAO_SEMANAL]
print('histórico semanal:', HIST.rodada.unique() if len(HIST) else 'nenhum (rode o backtest do Producao_Semanal ou use as pastas da pesquisa)')
arq_f = os.path.join(P['PS'], 'previsao_semanal_ultimo.parquet')
FUT = pd.read_parquet(arq_f) if os.path.exists(arq_f) else None
FUT = FUT[FUT.modelo == PL.CAMPEAO_SEMANAL] if FUT is not None else None
print('próximas semanas:', None if FUT is None else sorted(FUT.date.dt.date.unique()))


In [ ]:
# @title 3 · Previsões do campeão de temporada (produção + pesquisa)
fs = sorted(glob.glob(os.path.join(P['PT'], 'previsao_temporada_*.parquet')))
TEMP = pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True) if fs else pd.DataFrame(columns=L.COLS_PREV)
r7 = sorted(glob.glob(os.path.join(PASTA_DRIVE, 'resultados_uf_r7', 'uf', f'{PL.CAMPEAO_TEMPORADA}_s*.parquet')))
if r7:
    PESQ = pd.concat([pd.read_parquet(f) for f in r7], ignore_index=True)
    TEMP = pd.concat([TEMP, PESQ[~PESQ.season.isin(set(TEMP.season))]], ignore_index=True)
TEMP = TEMP[TEMP.modelo == PL.CAMPEAO_TEMPORADA].copy()
TEMP['date'] = pd.to_datetime(TEMP.date)
S_ULT = int(TEMP.season.max()) if len(TEMP) else None
print('temporadas:', sorted(TEMP.season.unique()))


In [ ]:
# @title 4 · Gráficos (Brasil, 5 regiões, 26 UFs) e o documento HTML
OBS = G.observado(D)
sem = (G.agregar_previsoes(HIST) if len(HIST) else None, G.agregar_previsoes(FUT) if FUT is not None and len(FUT) else None)
tem = G.agregar_previsoes(TEMP) if len(TEMP) else None
FIG = G.figuras_todas(OBS, os.path.join(P['REL'], 'figuras'), semanal=sem if any(x is not None for x in sem) else None, temporada=tem)
TS = PL.tabela_semanal(FUT, D) if FUT is not None else None
TS = TS.pivot_table(index='uf', columns='date', values='pred').round(0).reset_index() if TS is not None else None
if TS is not None:
    TS.columns = ['UF'] + [f'{pd.Timestamp(c):%d/%m/%Y}' for c in TS.columns[1:]]
TT = PL.resumo_temporada(TEMP, D, S_ULT) if S_ULT else None
arq = RC.gerar(os.path.join(P['REL'], 'modelos_campeoes.html'), FIG, TS, TT,
               dict(ultima_semana=f'{ULTIMA:%d/%m/%Y}', temporada=f'{S_ULT}/{S_ULT + 1}' if S_ULT else '–'))
print('documento:', arq, f'({os.path.getsize(arq) / 1e6:.1f} MB)')
from IPython.display import Image, display
for u in ('Brasil', 'Sudeste', 'SP', 'PR'):
    for k in ('semanal', 'temporada'):
        if FIG.get(u, {}).get(k):
            display(Image(FIG[u][k]))
